In [3]:
%pip install hypersync asyncio requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 15.2 MB/s eta 0:00:00


The below code can be used to get the real time OrderFilled event data. The data can be used to plot a histogram of the buy and sell orders.

In [29]:
import hypersync
import asyncio
import time # Import time for the sleep
from tqdm.asyncio import tqdm  # Import tqdm for progress bar

# returns all logs of a specific event from a contract within a block range

async def stream_events():
    # Create hypersync client using the mainnet hypersync endpoint (default)
    client = hypersync.HypersyncClient(hypersync.ClientConfig(
        url='https://polygon.hypersync.xyz'
    ))

    usdt_contract = "0x4bFb41d5B3570DeFd03C39a9A4D8dE6Bd8B8982E"

    # topic0 of transaction event signature (hash of event signature)
    # query will return logs of this event
    event_topic_0 = "0xd0a08e8c493f9c94f29311604c9de1b4e8c8d4c06bd0c789af57f2d65bfec0f6"

    # Start streaming from a recent block initially
    current_height = await client.get_height()
    start_block = current_height - 100 # Adjust the starting block as needed for initial fetch

    print(f"Starting stream from block {start_block}...")

    streamConfig = hypersync.StreamConfig()

    # Initialize the decoder outside the loop
    decoder = hypersync.Decoder(
      ["OrderFilled(bytes32 indexed orderHash,address indexed maker,address indexed taker, uint256 makerAssetId, uint256 takerAssetId, uint256 makerAmountFilled, uint256 takerAmountFilled, uint256 fee)"]
    )


    while True:
        query = hypersync.preset_query_logs_of_event(usdt_contract, event_topic_0, start_block)

        # Start the stream
        receiver = await client.stream(query, streamConfig)

        print(f"Streaming events from block {start_block}...")
        while True:
            res = await receiver.recv()
            # If res is None, it means the current stream chunk has finished
            if res is None:
                print("Current stream chunk finished. Checking for new blocks...")
                # Update the start_block to the next block to continue streaming from
                # Need to get the latest height before querying again
                latest_height = await client.get_height()
                if start_block < latest_height:
                    start_block = latest_height
                    break # Break the inner loop to start a new stream from the new block
                else:
                    # If no new blocks, wait a bit before checking again
                    print("No new blocks yet. Waiting...")
                    await asyncio.sleep(5) # Wait for 5 seconds before re-querying
                    continue # Continue the inner loop to poll again from the same start block


            if res.data and res.data.logs:
                print(f"Received a chunk of {len(res.data.logs)} events.")
                # Decode the logs before printing
                decoded_logs = await decoder.decode_logs(res.data.logs)
                for log in res.data.logs:
                  print(f"Decoded Topics: {log.topics}")
                for log in decoded_logs:
                  # Access decoded values using log.body and log.topics
                  print(f"Decoded Log Data: {log.body}")


            # Update the start_block to the next block for the next poll within the same stream
            if res.next_block:
                 start_block = res.next_block


        # Add a small delay before starting the next stream from the new block height
        await asyncio.sleep(1)


await stream_events()

Streaming output truncated to the last 5000 lines.
Decoded Topics: ['0xd0a08e8c493f9c94f29311604c9de1b4e8c8d4c06bd0c789af57f2d65bfec0f6', '0xc5296ec51cda8a65c79ab7e301c0dcb91694dc6425a22e5e07a17090b2cfd2b5', '0x000000000000000000000000a540e7ff75ef4bf7ebc6c793cbabe87cee970b43', '0x0000000000000000000000004bfb41d5b3570defd03c39a9a4d8de6bd8b8982e']
Decoded Topics: ['0xd0a08e8c493f9c94f29311604c9de1b4e8c8d4c06bd0c789af57f2d65bfec0f6', '0x9b85e7789a07d3c4db2dc400d38fbf5c3023d28777470a65474b8bd0631c6c99', '0x0000000000000000000000000d3b10b8eac8b089c6e4a695e65d8e044167c46b', '0x00000000000000000000000083b9f9e2d28ab875b9c9f73cdd5535c4c8926124']
Decoded Topics: ['0xd0a08e8c493f9c94f29311604c9de1b4e8c8d4c06bd0c789af57f2d65bfec0f6', '0xbdcc0ef50ab289661f73674c42dd38f34ff024ab730610a84f82325478292cb7', '0x00000000000000000000000083b9f9e2d28ab875b9c9f73cdd5535c4c8926124', '0x0000000000000000000000004bfb41d5b3570defd03c39a9a4d8de6bd8b8982e']
Decoded Topics: ['0xd0a08e8c493f9c94f29311604c9de1b4e8c8d4

CancelledError: 

Fetching a user's data from his address

In [ ]:
import requests

# Data API endpoint
data_api_url = "https://data-api.polymarket.com"

# Get current positions for a user
user_address = "0xdfd567f789411109eeed945cad28393a85c726e2"  # User's wallet address
response = requests.get(f"{data_api_url}/positions?user={user_address}")
positions = response.json()
print(positions)

[{'proxyWallet': '0xdfd567f789411109eeed945cad28393a85c726e2', 'asset': '65978063554798734083668439670689753780747884583014042762905590684207022599946', 'conditionId': '0xfc5be9b4d8a1c0af85cbbe664af68c4b3a8f03967228e356ed238b566f25cd2f', 'size': 52368.390086, 'avgPrice': 0.010184, 'initialValue': 533.319684635824, 'currentValue': 26.184195043000003, 'cashPnl': -507.13548959282406, 'percentPnl': -95.09033778476041, 'totalBought': 52368.390086, 'realizedPnl': 0, 'percentRealizedPnl': -95.09033778476041, 'curPrice': 0.0005, 'redeemable': False, 'mergeable': False, 'title': 'Will the Government shutdown end October 15-18', 'slug': 'will-the-government-shutdown-end-october-15-18', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/when-will-the-government-shutdown-end-kCiQ5ftNpjc-.jpg', 'eventId': '55056', 'eventSlug': 'when-will-the-government-shutdown-end-545', 'outcome': 'Yes', 'outcomeIndex': 0, 'oppositeOutcome': 'No', 'oppositeAsset': '14081421389829657542054344002504314829

getting active markets sorted by volume in last 24 hrs in descending order


In [31]:
import requests

# Gamma API endpoint
url = "https://gamma-api.polymarket.com/markets"

# Parameters to get active markets sorted by volume
params = {
    "closed": "false",      # Only active markets
    "order": "volume24hr",  # Sort by 24-hour volume
    "ascending": "false",   # Highest volume first
    "limit": 10             # Get top 10
}

# Make the request
response = requests.get(url, params=params)
markets = response.json()
top_10_market_by_vol24hr = [{'question': market['question'], 'tokenIDs': market['clobTokenIds'], 'conditionId': market['conditionId']} for market in markets]

In [32]:
top_10_market_by_vol24hr

[{'question': 'Will Frances Fitzgerald win the Irish Presidential Election?',
  'tokenIDs': '["13178519333823993662934968430857776162550570445253471722003513716922201689105", "47113635093830865822807939725855960883737645711087828564134442752879834974569"]',
  'conditionId': '0x4037ced85ebd734a30727e948abfe77b19fc2ce7ee0560a933d0c080fe076516'},
 {'question': 'Fed increases interest rates by 25+ bps after October 2025 meeting?',
  'tokenIDs': '["60156625646569576964584621291475860303247825215466310280401996727843189574431", "75662380910516595836713877605277832183886163391379455962356909608435996726535"]',
  'conditionId': '0xd6e0317cee513cc3afdc1954ac576b241ad2287536e88b17fd3a77b4c3d0976e'},
 {'question': 'Xi Jinping out in 2025?',
  'tokenIDs': '["114304586861386186441621124384163963092522056897081085884483958561365015034812", "112744882674787019048577842008042029962234998947364561417955402912669471494485"]',
  'conditionId': '0xf2ce8d3897ac5009a131637d3575f1f91c579bd08eecce6ae2b2da0f32

now we know the condition id and its corresponding unique token ids (first one is for YES share and second one for NO share) we get the top limit holders of that market


In [ ]:
# Data API endpoint
data_api_url = "https://data-api.polymarket.com"
limit = 5

all_holders = {}

for market in top_10_market_by_vol24hr:
    condition_id = market['conditionId']
    response = requests.get(f"{data_api_url}/holders?market={condition_id}&limit={limit}")
    holders = response.json()
    all_holders[condition_id] = holders
    print(f"Top holders for market {market['question']}:")
    for holder in holders:
        print(holder)

Top holders for market Will Frances Fitzgerald win the Irish Presidential Election?:
{'token': '13178519333823993662934968430857776162550570445253471722003513716922201689105', 'holders': [{'proxyWallet': '0xc47313703ca556b0a303c2f33717a83f31ea5908', 'bio': '', 'asset': '13178519333823993662934968430857776162550570445253471722003513716922201689105', 'pseudonym': 'Wooden-Chili', 'amount': 59510, 'displayUsernamePublic': True, 'outcomeIndex': 0, 'name': '0xc47313703CA556b0A303c2f33717A83f31ea5908-1760861748957', 'profileImage': '', 'profileImageOptimized': ''}, {'proxyWallet': '0xd218e474776403a330142299f7796e8ba32eb5c9', 'bio': 'broken grid', 'asset': '13178519333823993662934968430857776162550570445253471722003513716922201689105', 'pseudonym': 'Attentive-Crewman', 'amount': 51168.69852, 'displayUsernamePublic': True, 'outcomeIndex': 0, 'name': 'cigarettes', 'profileImage': '', 'profileImageOptimized': ''}, {'proxyWallet': '0x985c7c2d9b8d9a935658d21f46a740a3267c6611', 'bio': '', 'asset': 